# 2. Разметка с нуля
Читаем датасет, создаём проект и загружаем картинки с перекрытием 3. Исполнитель рисует все рамки.

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from seminar.data import read_json, write_json
from seminar.api import request, all_items, upload_tasks, classic_view, boxes_to_annotations

load_dotenv()
Path("results").mkdir(exist_ok=True)

### Датасет

In [ ]:
images = read_json("data/dataset.json")
pd.DataFrame(images)[["image_id", "image_path", "n_boxes"]].head()

### Проект
Используем проект 10670 и пул 6756373. Для создания нового проекта и пула задайте оба `EXISTING_*_ID = None`.

In [ ]:
project_payload = {
    "public_name": "Выделите весь транспорт на фотографии [MANUAL]",
    "public_description": "Выделите объекты прямоугольниками. Если рамки уже есть, проверьте и исправьте их.",
    "public_instructions": Path("interface/instructions.html").read_text(encoding="utf-8"),
    "task_spec": {
        "input_spec": {
            "image": {"type": "url", "required": True},
            "image_id": {"type": "string", "required": True, "hidden": True},
            "initial_boxes": {"type": "json", "required": True, "hidden": False},
        },
        "output_spec": {
            "result": {"type": "json", "required": True},
            "preannotation_initialized": {"type": "boolean", "required": False},
        },
        "view_spec": classic_view("task"),
    },
    "assignments_issuing_type": "AUTOMATED",
    "assignments_automerge_enabled": False,
}

In [ ]:
EXISTING_PROJECT_ID = "10670"
EXISTING_POOL_ID = "6756373"

if EXISTING_POOL_ID and not EXISTING_PROJECT_ID:
    raise ValueError("Для существующего пула укажите EXISTING_PROJECT_ID")

project = (request("GET", f"projects/{EXISTING_PROJECT_ID}") if EXISTING_PROJECT_ID
           else request("POST", "projects", json=project_payload))
project_id = project["id"]
project_id


### Пул

In [ ]:
REWARD = 1
OVERLAP = 3

pool_payload = {
    "project_id": project_id,
    "private_name": "manual",
    "may_contain_adult_content": False,
    "reward_per_assignment": REWARD,
    "assignment_max_duration_seconds": 1800,
    "will_expire": (datetime.now(timezone.utc) + timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "auto_accept_solutions": True,
    "defaults": {"default_overlap_for_new_task_suites": OVERLAP},
    "mixer_config": {"real_tasks_count": 1, "golden_tasks_count": 0, "training_tasks_count": 0},
    "filter": {"and": [
        {"category": "computed", "key": "client_type", "operator": "EQ", "value": "BROWSER"},
        {"category": "profile", "key": "languages", "operator": "IN", "value": "RU"},
    ]},
}

In [ ]:
pool = (request("GET", f"pools/{EXISTING_POOL_ID}") if EXISTING_POOL_ID
        else request("POST", "pools", json=pool_payload))
assert str(pool["project_id"]) == str(project_id), "Пул принадлежит другому проекту"
pool_id = pool["id"]
REWARD = pool["reward_per_assignment"]
OVERLAP = pool["defaults"]["default_overlap_for_new_task_suites"]
pool_id


### Задания

In [ ]:
tasks = [
    {"pool_id": pool_id, "overlap": OVERLAP,
     "input_values": {"image": row["image_url"], "image_id": row["image_id"], "initial_boxes": []}}
    for row in images
]
len(tasks), len(tasks) * OVERLAP * REWARD

In [ ]:
if EXISTING_POOL_ID:
    print("Задания уже в пуле:", len(all_items("tasks", pool_id=pool_id)))
else:
    upload_tasks(pool_id, tasks)


### Модерация

In [ ]:
validation_url = f"https://tasks.yandex.ru/api/new/requester/pools/{pool_id}/validate"
validation = request("GET", validation_url)
validation

Если активен `MUST_PASS_MODERATION`, отправим пул на модерацию.

In [ ]:
if EXISTING_POOL_ID:
    print("Этот пул уже отправлен на модерацию; статус — в предыдущей ячейке.")
elif any(item["type"] == "MUST_PASS_MODERATION" and item.get("active") for item in validation):
    moderation = request(
        "PUT",
        f"https://tasks.yandex.ru/api/new/requester/poolModeration/{pool_id}",
        json={"status": "READY"},
    )
    print(moderation)
else:
    print("Модерация не требуется или уже пройдена")

### Запуск
После прохождения модерации выполните следующую ячейку.

In [ ]:
validation = request("GET", validation_url)
blockers = [item for item in validation if item.get("active") and item.get("blocker", True)]
if blockers:
    raise ValueError(f"Пул пока нельзя открыть: {blockers}")
request("POST", f"pools/{pool_id}/open")

### Результаты
Забираем результаты.

In [ ]:
assignments = all_items("assignments", pool_id=pool_id)
write_json("results/manual.json", assignments)
pd.Series([a["status"] for a in assignments]).value_counts()